# Dashboard Visualisasi — Rasio Dosen:Mahasiswa Universitas Siliwangi
**Sumber:** `Data/Processed/master_looker_unsil.csv` (output ETL)
**Periode:** Ganjil 2023 — Ganjil 2025

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
PATH_MASTER = os.path.join(ROOT, 'Data', 'Processed', 'master_looker_unsil.csv')
PATH_VIZ    = os.path.join(ROOT, 'Outputs', 'Visualizations')
os.makedirs(PATH_VIZ, exist_ok=True)

# Style
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11,
                     'axes.titlesize': 13, 'axes.titleweight': 'bold',
                     'figure.dpi': 120})
COLORS = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd']
PERIOD_ORDER = ['Ganjil 2023','Genap 2023','Ganjil 2024','Genap 2024','Ganjil 2025']

df = pd.read_csv(PATH_MASTER)
df['tahun_pelaporan'] = pd.Categorical(df['tahun_pelaporan'], categories=PERIOD_ORDER, ordered=True)
df = df.sort_values('tahun_pelaporan')
print(f'Data loaded: {len(df)} baris | {df["nama_program_studi"].nunique()} prodi | {df["tahun_pelaporan"].nunique()} periode')
display(df.head(3))

---
## 1. Visualisasi Tingkat Institusi

In [ ]:
# Agregasi institusi per periode
inst = df.groupby('tahun_pelaporan', observed=True).agg(
    total_mahasiswa=('jumlah_mahasiswa','sum'),
    total_dosen=('total_dosen','sum'),
    rata_rasio=('nilai_rasio','mean')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Tren Tingkat Institusi — Universitas Siliwangi', fontsize=15, fontweight='bold', y=1.02)

# A. Line chart tren rasio
ax = axes[0]
ax.plot(inst['tahun_pelaporan'], inst['rata_rasio'], marker='o', color=COLORS[0], linewidth=2.5, markersize=8)
ax.axhline(y=45, color='red', linestyle='--', linewidth=1.5, label='Batas Dikti (1:45)')
ax.set_title('Tren Rata-Rata Rasio Dosen:Mahasiswa')
ax.set_ylabel('Nilai Rasio (1:x)')
ax.set_xticklabels(inst['tahun_pelaporan'], rotation=30, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)
for i, (x, y) in enumerate(zip(range(len(inst)), inst['rata_rasio'])):
    ax.annotate(f'{y:.1f}', (i, y), textcoords='offset points', xytext=(0,8), ha='center', fontsize=9)

# B. Bar chart total mahasiswa
ax = axes[1]
bars = ax.bar(inst['tahun_pelaporan'], inst['total_mahasiswa'], color=COLORS[1], edgecolor='white', linewidth=0.5)
ax.set_title('Total Mahasiswa Aktif per Semester')
ax.set_ylabel('Jumlah Mahasiswa')
ax.set_xticklabels(inst['tahun_pelaporan'], rotation=30, ha='right')
ax.grid(True, axis='y', alpha=0.3)
for bar, v in zip(bars, inst['total_mahasiswa']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50, f'{int(v):,}', ha='center', va='bottom', fontsize=9)

# C. Bar chart total dosen
ax = axes[2]
bars = ax.bar(inst['tahun_pelaporan'], inst['total_dosen'], color=COLORS[2], edgecolor='white', linewidth=0.5)
ax.set_title('Total Dosen Tetap per Semester')
ax.set_ylabel('Jumlah Dosen')
ax.set_xticklabels(inst['tahun_pelaporan'], rotation=30, ha='right')
ax.grid(True, axis='y', alpha=0.3)
for bar, v in zip(bars, inst['total_dosen']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{int(v):,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(PATH_VIZ, 'viz_institusi.png'), bbox_inches='tight', dpi=150)
plt.show()
print('[TABEL] Ringkasan Tingkat Institusi:')
display(inst)

---
## 2. Visualisasi Tingkat Program Studi

In [ ]:
# Heatmap rasio per prodi × per semester
pivot = df.pivot_table(index='nama_program_studi', columns='tahun_pelaporan',
                       values='nilai_rasio', aggfunc='mean', observed=True)
pivot = pivot.reindex(columns=PERIOD_ORDER)
pivot = pivot.sort_values(PERIOD_ORDER[-1], ascending=False)

fig, ax = plt.subplots(figsize=(12, max(8, len(pivot)*0.4)))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r', linewidths=0.3,
            cbar_kws={'label':'Nilai Rasio (1:x)'}, ax=ax,
            vmin=0, vmax=45)
ax.set_title('Heatmap Rasio Dosen:Mahasiswa per Program Studi × Semester\nUniversitas Siliwangi',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Periode Pelaporan', fontweight='bold')
ax.set_ylabel('Program Studi', fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(PATH_VIZ, 'heatmap_prodi_semester.png'), bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Bar chart perbandingan rasio antar prodi - periode terbaru
latest = PERIOD_ORDER[-1]
df_latest = df[df['tahun_pelaporan']==latest].copy()
df_latest = df_latest.groupby('nama_program_studi', observed=True)['nilai_rasio'].mean().reset_index()
df_latest = df_latest.sort_values('nilai_rasio', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(8, len(df_latest)*0.35)))
colors = ['#d62728' if v > 45 else '#1f77b4' for v in df_latest['nilai_rasio']]
bars = ax.barh(df_latest['nama_program_studi'], df_latest['nilai_rasio'], color=colors, edgecolor='white')
ax.axvline(x=45, color='red', linestyle='--', linewidth=2, label='Batas Dikti (1:45)')
ax.set_title(f'Perbandingan Rasio Dosen:Mahasiswa per Prodi\nPeriode {latest}', fontweight='bold')
ax.set_xlabel('Nilai Rasio (1:x)')
ax.legend()
ax.grid(True, axis='x', alpha=0.3)
for bar, v in zip(bars, df_latest['nilai_rasio']):
    ax.text(v+0.2, bar.get_y()+bar.get_height()/2, f'{v:.1f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PATH_VIZ, 'bar_rasio_prodi_terbaru.png'), bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Line chart tren 5 prodi rasio tertinggi vs 5 terendah (periode terbaru)
top5  = df_latest.nlargest(5, 'nilai_rasio')['nama_program_studi'].tolist()
bot5  = df_latest.nsmallest(5, 'nilai_rasio')['nama_program_studi'].tolist()
sel   = top5 + bot5

df_sel = df[df['nama_program_studi'].isin(sel)].copy()
df_sel = df_sel.groupby(['nama_program_studi','tahun_pelaporan'], observed=True)['nilai_rasio'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, group, title, clist in zip(axes, [top5, bot5],
    ['5 Prodi Rasio TERTINGGI (Beban Terberat)','5 Prodi Rasio TERENDAH (Beban Teringan)'],
    [plt.cm.Reds(np.linspace(0.4,0.9,5)), plt.cm.Blues(np.linspace(0.4,0.9,5))]):
    for i, prodi in enumerate(group):
        sub = df_sel[df_sel['nama_program_studi']==prodi]
        ax.plot(sub['tahun_pelaporan'].astype(str), sub['nilai_rasio'],
                marker='o', label=prodi, color=clist[i], linewidth=2)
    ax.axhline(45, color='red', linestyle='--', linewidth=1.5, label='Batas Dikti')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Nilai Rasio (1:x)')
    ax.set_xticklabels([p for p in PERIOD_ORDER], rotation=30, ha='right')
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)

plt.suptitle('Tren Rasio Dosen:Mahasiswa — Perbandingan Prodi Ekstrem\nUniversitas Siliwangi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PATH_VIZ, 'line_tren_top5_bot5.png'), bbox_inches='tight', dpi=150)
plt.show()

---
## 3. Tabel Analisis untuk Pembahasan

In [ ]:
# Tabel ranking prodi berdasarkan rasio periode terbaru
latest = PERIOD_ORDER[-1]
ranking = df[df['tahun_pelaporan']==latest].groupby('nama_program_studi', observed=True).agg(
    jenjang=('jenjang','first'),
    akreditasi=('akreditasi_prodi','first'),
    total_dosen=('total_dosen','mean'),
    jumlah_mahasiswa=('jumlah_mahasiswa','mean'),
    nilai_rasio=('nilai_rasio','mean')
).reset_index()
ranking['total_dosen'] = ranking['total_dosen'].round(0).astype(int)
ranking['jumlah_mahasiswa'] = ranking['jumlah_mahasiswa'].round(0).astype(int)
ranking['nilai_rasio'] = ranking['nilai_rasio'].round(2)
ranking['rasio_fmt'] = ranking['nilai_rasio'].apply(lambda x: f'1:{x:.2f}')
ranking['status_dikti'] = ranking['nilai_rasio'].apply(lambda x: '⚠ MELEBIHI BATAS' if x > 45 else '✅ Normal')
ranking = ranking.sort_values('nilai_rasio', ascending=False).reset_index(drop=True)
ranking.index += 1
ranking.index.name = 'Ranking'
print(f'[TABEL 5] Ranking Prodi berdasarkan Rasio — Periode {latest}:')
display(ranking[['nama_program_studi','jenjang','akreditasi','total_dosen','jumlah_mahasiswa','rasio_fmt','status_dikti']])
print(f'\nJumlah prodi dengan rasio > 1:45: {(ranking["nilai_rasio"] > 45).sum()} prodi')

In [ ]:
# Tabel prodi yang melebihi batas wajar Dikti (> 1:45)
pelanggaran = ranking[ranking['nilai_rasio'] > 45][['nama_program_studi','jenjang','total_dosen','jumlah_mahasiswa','rasio_fmt']].copy()
pelanggaran = pelanggaran.rename(columns={'nama_program_studi':'Program Studi','jenjang':'Jenjang',
                                          'total_dosen':'Dosen','jumlah_mahasiswa':'Mahasiswa','rasio_fmt':'Rasio'})
print(f'[TABEL 6] Program Studi yang Melebihi Batas Rasio Dikti (> 1:45) — Periode {latest}:')
if len(pelanggaran) == 0:
    print('Tidak ada prodi yang melebihi batas 1:45 pada periode ini.')
else:
    display(pelanggaran)
    print(f'Total: {len(pelanggaran)} prodi perlu perhatian.')

---
## 4. Dashboard Final (Semua Chart dalam 1 Tampilan)

In [ ]:
fig = plt.figure(figsize=(20, 22))
fig.patch.set_facecolor('#f8f9fa')
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

# Title
fig.suptitle('DASHBOARD ANALITIK\nRasio Dosen:Mahasiswa Universitas Siliwangi',
             fontsize=18, fontweight='bold', y=0.98)

# Panel 1: Line tren institusi
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(range(len(inst)), inst['rata_rasio'], marker='o', color='#1f77b4', linewidth=2.5, markersize=9)
ax1.axhline(45, color='red', linestyle='--', linewidth=1.5, label='Batas Dikti')
ax1.set_title('Tren Rata-Rata Rasio Institusi', pad=10)
ax1.set_xticks(range(len(inst))); ax1.set_xticklabels(PERIOD_ORDER, rotation=25, ha='right')
ax1.set_ylabel('Nilai Rasio (1:x)'); ax1.legend(fontsize=9); ax1.grid(alpha=0.3)
for i, v in enumerate(inst['rata_rasio']): ax1.annotate(f'{v:.1f}', (i,v), xytext=(0,8), textcoords='offset points', ha='center', fontsize=9)

# Panel 2: Bar mahasiswa
ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(range(len(inst)), inst['total_mahasiswa'], color='#ff7f0e', edgecolor='white')
ax2.set_title('Total Mahasiswa per Semester', pad=10)
ax2.set_xticks(range(len(inst))); ax2.set_xticklabels(PERIOD_ORDER, rotation=25, ha='right')
ax2.set_ylabel('Jumlah'); ax2.grid(alpha=0.3, axis='y')
for i, v in enumerate(inst['total_mahasiswa']): ax2.text(i, v+30, f'{int(v):,}', ha='center', fontsize=9)

# Panel 3: Bar dosen
ax3 = fig.add_subplot(gs[1, 0])
ax3.bar(range(len(inst)), inst['total_dosen'], color='#2ca02c', edgecolor='white')
ax3.set_title('Total Dosen per Semester', pad=10)
ax3.set_xticks(range(len(inst))); ax3.set_xticklabels(PERIOD_ORDER, rotation=25, ha='right')
ax3.set_ylabel('Jumlah'); ax3.grid(alpha=0.3, axis='y')
for i, v in enumerate(inst['total_dosen']): ax3.text(i, v+1, f'{int(v):,}', ha='center', fontsize=9)

# Panel 4: Bar rasio terbaru (top 10)
ax4 = fig.add_subplot(gs[1, 1])
top10 = df_latest.nlargest(10, 'nilai_rasio')
clrs  = ['#d62728' if v > 45 else '#1f77b4' for v in top10['nilai_rasio']]
ax4.barh(top10['nama_program_studi'], top10['nilai_rasio'], color=clrs)
ax4.axvline(45, color='red', linestyle='--', linewidth=1.5)
ax4.set_title('Top 10 Prodi Rasio Tertinggi (Terkini)', pad=10)
ax4.set_xlabel('Nilai Rasio (1:x)'); ax4.grid(alpha=0.3, axis='x')

# Panel 5: Heatmap (span bottom row)
ax5 = fig.add_subplot(gs[2, :])
pivot_dash = pivot.head(15)  # 15 prodi rasio tertinggi
sns.heatmap(pivot_dash, annot=True, fmt='.1f', cmap='RdYlGn_r', linewidths=0.3,
            cbar_kws={'label':'Rasio (1:x)', 'shrink':0.6}, ax=ax5, vmin=0, vmax=45)
ax5.set_title('Heatmap Rasio per Prodi × Semester (15 Tertinggi)', pad=10)
ax5.set_xlabel('Periode', fontweight='bold'); ax5.set_ylabel('')
ax5.tick_params(axis='x', rotation=20); ax5.tick_params(axis='y', rotation=0)

plt.savefig(os.path.join(PATH_VIZ, 'dashboard_final.png'), bbox_inches='tight', dpi=150, facecolor=fig.get_facecolor())
plt.show()
print('✅ Dashboard tersimpan:', os.path.join(PATH_VIZ, 'dashboard_final.png'))